
# 02 – Emotion / EPI Indices & Econometric Modelling

This notebook:

1. Loads the **financial returns + controls** panel from Notebook 1.
2. Loads the **Emotion Index (EI)** and **Emotion Polarity Index (EPI)** datasets (already computed).
3. Aligns the emotion indices with the financial data.
4. Creates **lagged EPI variables** for predictive modelling.
5. Estimates **ARX models with EPI** for each asset.
6. Estimates a **VARX model** with EPI as an exogenous driver.
7. Optionally saves a combined finance–emotion panel for further work.


In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.tsa.api import VAR

pd.options.display.float_format = "{:.4f}".format
plt.rcParams["figure.figsize"] = (10, 4)

DATA_DIR = "./data/"
OUTPUT_DIR = "./data/"

FINANCIAL_PANEL_FILE = DATA_DIR + "financial_returns_and_controls.parquet"

# Adjust these to your actual EI/EPI files
EMOTION_FILE = DATA_DIR + "emotion_indices_daily.parquet"  # or .csv




## 1. Load financial panel


In [ ]:

fin = pd.read_parquet(FINANCIAL_PANEL_FILE)
fin.index = pd.to_datetime(fin.index)
fin = fin.sort_index()

print(fin.head())
print(fin.info())



## 2. Load Emotion / EPI indices

This assumes a single file with columns like:

- `EI_creator`
- `EI_community`
- `EPI`

You can adjust the column names and file format as needed.


In [ ]:

# Try parquet, fall back to CSV
try:
    epi = pd.read_parquet(EMOTION_FILE)
except Exception:
    epi = pd.read_csv(EMOTION_FILE)

# Try to detect a date column
if "Date" in epi.columns:
    epi["Date"] = pd.to_datetime(epi["Date"])
    epi = epi.set_index("Date")
elif "date" in epi.columns:
    epi["date"] = pd.to_datetime(epi["date"])
    epi = epi.set_index("date")
else:
    # If the index is already datetime-like, we assume it's fine
    if not isinstance(epi.index, pd.DatetimeIndex):
        raise KeyError("No Date/date column found. Please adjust EMOTION_FILE loading.")

epi = epi.sort_index()

print(epi.head())
print(epi.info())



## 3. Basic checks and simple plots of EPI


In [ ]:

if "EPI" not in epi.columns:
    print("WARNING: 'EPI' column not found in emotion file. Available columns:", epi.columns.tolist())

# Example: 7-day moving average of EPI
if "EPI" in epi.columns:
    epi["EPI_ma7"] = epi["EPI"].rolling(7, min_periods=1).mean()
    ax = epi[["EPI", "EPI_ma7"]].plot(title="EPI and 7-day moving average")
    ax.set_xlabel("Date")
    ax.set_ylabel("Index level")
    plt.show()


## 4. Align emotion indices with financial data

We align EI/EPI to the financial business-day calendar by forward-filling from the last observed value (so weekend changes carry into Monday).

In [ ]:
# Align EI/EPI to the financial calendar (forward-fill to carry weekend values into Monday)
epi_aligned = epi.reindex(fin.index).ffill()
panel = fin.join(epi_aligned, how="left")

print("Aligned panel shape:", panel.shape)
panel.head()

In [ ]:
# Optional: strict overlap without forward-fill (uncomment if desired)
# panel = fin.join(epi, how="inner")
# print(panel.shape)


## 5. Create lagged EPI (and optionally EI) variables

We use lagged EPI values (1, 3, and 5 days) so Monday models can react to weekend moves (l1=Sun, l3=Fri).

In [ ]:

lags = [1, 3, 5]  # include weekend moves (Mon l1=Sun, l3=Fri)

if "EPI" not in panel.columns:
    raise KeyError("Expected an 'EPI' column in the merged panel. Please adjust to your emotion column names.")

for L in lags:
    panel[f"EPI_l{L}"] = panel["EPI"].shift(L)

# Optionally include lags of EI_creator / EI_community if present
for ei_col in ["EI_creator", "EI_community"]:
    if ei_col in panel.columns:
        for L in lags:
            panel[f"{ei_col}_l{L}"] = panel[ei_col].shift(L)

panel = panel.dropna()
panel.head()



## 6. Define modelling dataset

We select:

- Asset returns: `r_btc`, `r_gold`, `r_spx`
- Control variables: `dln_eurusd`, `dln_usdcny`, `dln_oil`, `dy10`
- EPI lags: `EPI_l1`, `EPI_l2`, `EPI_l3`, `EPI_l5`


In [ ]:

return_cols = ["r_btc", "r_gold", "r_spx"]
control_cols = ["dln_eurusd", "dln_usdcny", "dln_oil", "dy10"]
epi_lag_cols = [f"EPI_l{L}" for L in lags]

missing_cols = [c for c in return_cols + control_cols if c not in panel.columns]
if missing_cols:
    print("WARNING: These expected columns are missing from panel:", missing_cols)

data_cols = [c for c in return_cols + control_cols + epi_lag_cols if c in panel.columns]
data = panel[data_cols].dropna()

print(data.head())
print(data.info())



## 7. ARX models with EPI

We extend the ARX models to include lagged EPI terms.


In [ ]:

# Create return lags for each asset
p = 1
for asset in return_cols:
    for lag in range(1, p + 1):
        panel[f"{asset}_l{lag}"] = panel[asset].shift(lag)

# Rebuild modelling dataset including return lags
exog_cols_base = control_cols + epi_lag_cols
for asset in return_cols:
    exog_cols_base += [f"{asset}_l{lag}" for lag in range(1, p + 1)]

# We'll construct per-asset exog sets below
panel_arx = panel.dropna(subset=return_cols + control_cols + epi_lag_cols)


In [ ]:

def fit_arx_with_epi(panel, asset, p=1, control_cols=None, epi_lag_cols=None):
    if control_cols is None:
        control_cols = []
    if epi_lag_cols is None:
        epi_lag_cols = []

    df = panel.copy()
    # Ensure lag columns exist
    for lag in range(1, p + 1):
        col_lag = f"{asset}_l{lag}"
        if col_lag not in df.columns:
            df[col_lag] = df[asset].shift(lag)

    cols_lag = [f"{asset}_l{lag}" for lag in range(1, p + 1)]
    cols_exog = cols_lag + control_cols + epi_lag_cols

    df = df.dropna(subset=[asset] + cols_exog)
    y = df[asset]
    X = sm.add_constant(df[cols_exog])

    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    return model

arx_epi_results = {}
for asset in return_cols:
    model = fit_arx_with_epi(panel, asset, p=p, control_cols=control_cols, epi_lag_cols=epi_lag_cols)
    arx_epi_results[asset] = model
    print(f"\n=== ARX + EPI results for {asset} ===")
    print(model.summary().tables[1])



You can compare these results to the baseline ARX models from Notebook 1 by examining:

- The significance and signs of the EPI lag coefficients.
- Changes in R² and information criteria.



## 8. VARX model with EPI

We now estimate a VAR model on the three return series, with control variables and EPI lags as exogenous regressors.


In [ ]:

Y = panel[return_cols]
X_exog = panel[control_cols + epi_lag_cols]

varx_data = pd.concat([Y, X_exog], axis=1).dropna()
Y_var = varx_data[return_cols]
X_var = varx_data[control_cols + epi_lag_cols]

print("VARX dataset shape:", Y_var.shape)


In [ ]:

# Choose lag order p (you can also use select_order as in Notebook 1)
p_var = 1

var_model = VAR(Y_var)
var_results = var_model.fit(maxlags=p_var, trend="c", exog=X_var)

print(var_results.summary())



You can perform Wald tests on the joint significance of the EPI lags across all equations if desired.

For example (pseudo-code, adjust to your design matrix):

```python
# Example: joint test that all EPI_l* coefficients are zero in all equations
# (requires constructing an appropriate restriction matrix R and vector r)
```



## 9. Save combined finance–emotion panel

This can be reused for robustness checks or additional models.


In [ ]:

combined_panel_file = OUTPUT_DIR + "finance_emotion_panel.parquet"
panel.to_parquet(combined_panel_file)
print("Saved combined panel to:", combined_panel_file)



### Notebook 2 complete ✅

You now have:

- A merged finance–emotion panel.
- ARX models with EPI as an exogenous predictor.
- A VARX model capturing cross-asset dynamics with EPI as an external driver.

You can extend this notebook with:
- Topic-specific EPIs,
- Alternative lag structures,
- Out-of-sample forecasting and Diebold–Mariano tests.
